# Multithreading with Numba

**Optional deep dive, about 20 minutes.** Numba can parallelize independent loop iterations with `prange`. We will set a thread limit so the exercise does not silently consume every core on an HPC node.

In [ ]:
import os
import time
import numpy as np
import numba
from numba import jit, prange

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
n_threads = min(4, numba.get_num_threads())
numba.set_num_threads(n_threads)
print("Numba threads used:", numba.get_num_threads())

## 1. Parallelize independent loop iterations

The midpoint rule estimates the integral of `4 / (1 + x**2)` from 0 to 1, which equals $\pi$. Each loop step calculates its value independently, and Numba automatically combines the reduction sum across threads when using `prange`.

In [ ]:
@jit
def estimate_pi_serial(n_steps):
    total = 0.0
    step = 1.0 / n_steps
    for index in range(n_steps):
        x = (index + 0.5) * step
        total += 4.0 / (1.0 + x * x)
    return total * step


@jit(parallel=True)
def estimate_pi_parallel(n_steps):
    total = 0.0
    step = 1.0 / n_steps
    for index in prange(n_steps):
        x = (index + 0.5) * step
        total += 4.0 / (1.0 + x * x)
    return total * step

In [ ]:
n_steps = 2_000_000 if test_mode else 50_000_000

# Call both functions once before timing.
serial_result = estimate_pi_serial(n_steps)
parallel_result = estimate_pi_parallel(n_steps)
np.testing.assert_allclose(serial_result, np.pi, rtol=1e-10)
np.testing.assert_allclose(parallel_result, serial_result, rtol=1e-12)

started = time.perf_counter()
estimate_pi_serial(n_steps)
serial_time = time.perf_counter() - started

started = time.perf_counter()
estimate_pi_parallel(n_steps)
parallel_time = time.perf_counter() - started

print(f"Serial:   {serial_time:.3f} s")
print(f"Parallel: {parallel_time:.3f} s")
print(f"Speedup:  {serial_time / parallel_time:.2f}x")

## 2. Check parallel loop diagnostics

Numba provides `parallel_diagnostics(level=1)` to report which loops were parallelized. Inspect this report to confirm that Numba parallelized the intended `prange` loop.

In [ ]:
estimate_pi_parallel.parallel_diagnostics(level=1)

## Your turn: 2D row-wise moving average

**10 minutes.** Extend the moving average to a 2D matrix. Write a function `moving_average_2d_parallel` that smooths each row independently across its columns.

1. Decorate the function with `@jit(parallel=True)`.
2. Use `prange` for the outer row loop (`r in prange(rows)`).
3. Keep the inner column loop serial (`c in range(1, cols - 1)`).
4. Verify correctness against `moving_average_2d_serial` using `np.testing.assert_allclose`.
5. Time serial vs parallel Numba.

```python
@jit(parallel=True)
def moving_average_2d_parallel(matrix):
    rows, cols = matrix.shape
    out = np.empty((rows, cols - 2), dtype=matrix.dtype)
    # Add your prange row loop and inner column loop
    return out
```

Blue sticky note means you need help. Yellow means you are ready to discuss.

In [ ]:
# Write and test your solution here before opening the solution cell below.

<details><summary>Solution and discussion</summary>

Run the solution cell below after attempting the exercise. Using `prange` on the outer row loop distributes independent rows across available worker threads.

```python
@jit(parallel=True)
def moving_average_2d_parallel(matrix):
    rows, cols = matrix.shape
    out = np.empty((rows, cols - 2), dtype=matrix.dtype)
    for r in prange(rows):
        for c in range(1, cols - 1):
            out[r, c - 1] = (matrix[r, c - 1] + matrix[r, c] + matrix[r, c + 1]) / 3.0
    return out
```

</details>

In [ ]:
@jit
def moving_average_2d_serial(matrix):
    rows, cols = matrix.shape
    out = np.empty((rows, cols - 2), dtype=matrix.dtype)
    for r in range(rows):
        for c in range(1, cols - 1):
            out[r, c - 1] = (matrix[r, c - 1] + matrix[r, c] + matrix[r, c + 1]) / 3.0
    return out

@jit(parallel=True)
def moving_average_2d_parallel(matrix):
    rows, cols = matrix.shape
    out = np.empty((rows, cols - 2), dtype=matrix.dtype)
    for r in prange(rows):
        for c in range(1, cols - 1):
            out[r, c - 1] = (matrix[r, c - 1] + matrix[r, c] + matrix[r, c + 1]) / 3.0
    return out

rng = np.random.default_rng(2026)
mat = rng.random((2000, 2000))

# Warmup and correctness check
res_serial = moving_average_2d_serial(mat)
res_parallel = moving_average_2d_parallel(mat)
np.testing.assert_allclose(res_parallel, res_serial, rtol=1e-12)

started = time.perf_counter()
moving_average_2d_serial(mat)
t_serial = time.perf_counter() - started

started = time.perf_counter()
moving_average_2d_parallel(mat)
t_parallel = time.perf_counter() - started

print(f"Serial Numba:   {t_serial * 1e3:.2f} ms")
print(f"Parallel Numba: {t_parallel * 1e3:.2f} ms")
print(f"Speedup:        {t_serial / t_parallel:.2f}x")

## Takeaway

- Use `prange` only when loop iterations are independent or accumulate into a reduction.
- Inspect `parallel_diagnostics(level=1)` to verify that Numba parallelized the loop.
- Always set thread limits (`numba.set_num_threads`) on multi-core HPC systems to avoid oversubscribing CPUs.